<a href="https://colab.research.google.com/github/hmmnyamminji/DL/blob/main/day21_practice2_RAG_%ED%8C%8C%EC%9D%B4%ED%94%84%EB%9D%BC%EC%9D%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# RAG (Retrieval-Augmented Generation, 검색 증강 생성) 파이프라인
# 문서 청킹 - 긴 규정집을 검색 단위로 자르기
# 검색 결과를 프롬포트에 '증거'로 주입

In [2]:
import torch
import torch.nn.functional as F
import requests, os, re


In [3]:
# 시청의 도로 규정 : knowledge/ 폴더의 규정 문서들 사용
documents = [
    "심각도 4 이상의 포트홀은 접수 후 24시간 이내에 긴급 보수팀을 배정한다.",
    "심각도 3 이하의 도로 균열은 주간 정기 점검 일정에 포함하여 7일 이내 처리한다.",
    "맨홀 뚜껑 이상 신고는 안전 펜스를 우선 설치하고 상수도사업소에 이관한다.",
    "낙하물 신고는 2차 사고 예방을 위해 즉시 현장 출동을 원칙으로 한다.",
    "처리 완료된 신고는 사진과 함께 신고자에게 앱 알림으로 통보한다.",
    "야간(22시~06시) 접수 신고는 익일 오전 담당자 배정 후 처리를 시작한다.",
]
print(f"knowledge/: 규정 {len(documents)}개")

knowledge/: 규정 6개


In [4]:
# 1) 사전 만들기: 글자 2조각 → 번호, n-gram 임베딩
def make_vocab(texts, n=2):
  grams = set()
  for t in texts:
    t = re.sub(r"\s+", "", t) # \s: 공백 제거
    grams |= {t[i:i+n] for i in range(len(t) - n + 1)} # |=: 합집합, 조각 시작 위치, 모든 2글자 조각을 모은다.
  return {g: i for i, g in enumerate(sorted(grams))}

vocab = make_vocab(documents)
print(vocab)

# 2) 문장 → 벡터 변환
def embed(text, vocab, n=2):
    v = torch.zeros(len(vocab))
    t = re.sub(f"\n", "", text) # 공백 없는 문장
    for i in range(len(t) - n + 1): # 2글자씩
      g = t[i:i+n]
      if g in vocab:
        v[vocab[g]] += 1 # 단어 등장 횟수 증가
    return v

# API 호출 - openAI 호환 형식으로 작성
VISION_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai"
VISION_MODELS = ["gemini-flash-lite-latest"] # Google이 업데이트하는 Flash-Lite 라인업의 가장 최신 버전으로 자동 연결

# API 키 - Colab Secrets 에서 가져오기
def _get_api_key():
  try:
    from google.colab import userdata
    API_KEY = userdata.get('VISION_API_KEY')
  except Exception:
    API_KEY = os.environ.get("VISION_API_KEY","") # Colab이 아니거나 Secret 이 없을 때 환경변수 딕셔너리에서 꺼냄
  return API_KEY

API_KEY = _get_api_key()

def call_llm(messages, temperature=0.2):
  if not API_KEY:
    return None
  try:
    r = requests.post(f"{VISION_BASE_URL}/chat/completions",
        headers={"Authorization": f"Bearer {API_KEY}"},
        json = {"model": VISION_MODELS[0], "messages":messages, "temperature": temperature},
        timeout=60)
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"]
  except Exception:
    return None

{'(2': 0, ')접': 1, '06': 2, '22': 3, '24': 4, '2시': 5, '2차': 6, '3이': 7, '4시': 8, '4이': 9, '6시': 10, '7일': 11, '~0': 12, '각도': 13, '간(': 14, '간이': 15, '간정': 16, '검일': 17, '게앱': 18, '고는': 19, '고상': 20, '고예': 21, '고자': 22, '과함': 23, '관한': 24, '균열': 25, '급보': 26, '기점': 27, '긴급': 28, '껑이': 29, '께신': 30, '낙하': 31, '내에': 32, '내처': 33, '는2': 34, '는사': 35, '는안': 36, '는익': 37, '다.': 38, '담당': 39, '당자': 40, '도3': 41, '도4': 42, '도로': 43, '도사': 44, '동을': 45, '된신': 46, '뚜껑': 47, '로균': 48, '로통': 49, '로한': 50, '료된': 51, '를시': 52, '를우': 53, '리를': 54, '리완': 55, '리한': 56, '림으': 57, '맨홀': 58, '물신': 59, '방을': 60, '배정': 61, '보수': 62, '보한': 63, '사고': 64, '사업': 65, '사진': 66, '상수': 67, '상신': 68, '상의': 69, '선설': 70, '설치': 71, '소에': 72, '수도': 73, '수신': 74, '수팀': 75, '수후': 76, '스를': 77, '시)': 78, '시~': 79, '시간': 80, '시작': 81, '시현': 82, '신고': 83, '심각': 84, '안전': 85, '알림': 86, '앱알': 87, '야간': 88, '업소': 89, '에게': 90, '에긴': 91, '에이': 92, '에포': 93, '여7': 94, '열은': 95, '예방': 96, '오전': 97, '완료': 98, '우선': 99, '원칙': 100

In [11]:
# 셀 1. 청킹 - 규정집단 통째로 못 넣는다. 분단 단위로 잘라(청크) '필요한 조각만' 검색해 넣는다.

규정집 = """제1조(긴급 보수) 심각도 4 이상의 포트홀은 접수 후 24시간 이내에 긴급 보수팀을 배정한다. 배정 결과는 신고자에게 통보한다.

제2조(정기 처리) 심각도 3 이하의 도로 균열은 주간 정기 점검 일정에 포함하여 7일 이내 처리한다.

제3조(맨홀) 맨홀 뚜껑 이상 신고는 안전 펜스를 우선 설치하고 상수도사업소에 이관한다. 이관 후에도 처리 현황을 추적한다.

제4조(낙하물) 낙하물 신고는 2차 사고 예방을 위해 즉시 현장 출동을 원칙으로 한다.

제5조(야간 접수) 야간(22시~06시) 접수 신고는 익일 오전 담당자 배정 후 처리를 시작한다. 단, 심각도 5는 야간에도 즉시 출동한다.

제6조(완료 통보) 처리 완료된 신고는 사진과 함께 신고자에게 앱 알림으로 통보한다."""

chunks = [c.strip() for c in 규정집.split("\n\n")] # 문단 기준으로 자름
print(f"청킹: 규정집 → {len(chunks)}개 조각 (분단 단위)")
vocab = make_vocab(chunks)
chunk_vectors = torch.stack([embed(c, vocab) for c in chunks])

청킹: 규정집 → 6개 조각 (분단 단위)


In [17]:
# 셀 2. RAG의 3단계 : 검색 → 증거 주입 → 생성

# 1) 검색(Retrieve): 질문과 비슷한 조각 k개 찾기
def retrieve(query, k=2):
  q = embed(query, vocab)  # 임베딩 → 벡터화
  sims = F.cosine_similarity(q.unsqueeze(0), chunk_vectors)  # (157,) → (1,157), 질문 1개 vs 문서 6개를 '한 번에' 유사도 검사 (6,)
  top = sims.argsort(descending=True)[:k] # 내림차순으로 정렬 상위 k개 '인덱스'
  return [(chunks[i], sims[i].item()) for i in top]

# 2) 프롬포트 템플릿
RAG_PROMPT = """ 너는 도로 관리 규정 안내 담당자다.
아래 [규정] 에 근거해서만 답하라. 규정에 없는 내용은 "규정에 없음"이라고 답하라.

[규정]
{context}

[질문] {question}
[답변] 근거 조항을 인용하며 두 문장 이내로."""

# 3) RAG 전체 흐름: 검색 → 주입 → 생성
def raw_answer(question):
  evidence = retrieve(question, k=2) # 1. 검색 결과
  context = "\n".join(str(e) for e in evidence) # Top K 검색 결과를 한 문장으로
  prompt = RAG_PROMPT.format(context=context, question=question) # 질문과 검색 결과를 프롬포트에 주입, str.format() 값을 채워서 새 문자열 반환
  answer = call_llm([{"role": "user", "content": prompt}])
  return answer, evidence

# 4) 실행
question = "밤에 신고하면 언제 처리 시작돼요?"
answer, evidence = raw_answer(question)
print(f"\n질문: {question}")
print("검색된 증거:")
for e in evidence: print(f" , {e}...")
print(f"답변 {answer}")


질문: 밤에 신고하면 언제 처리 시작돼요?
검색된 증거:
 , ('제6조(완료 통보) 처리 완료된 신고는 사진과 함께 신고자에게 앱 알림으로 통보한다.', 0.3061861991882324)...
 , ('제5조(야간 접수) 야간(22시~06시) 접수 신고는 익일 오전 담당자 배정 후 처리를 시작한다. 단, 심각도 5는 야간에도 즉시 출동한다.', 0.23354968428611755)...
답변 제5조(야간 접수)에 따라, 야간(22시~06시) 접수 신고는 익일 오전 담당자 배정 후 처리를 시작합니다. 단, 심각도 5는 야간에도 즉시 출동합니다.


In [13]:
# 셀 3. 대조 실행 - LLM 단독 vs RAG
def _llm_solo(question): # LLM 단독 - 규정 없이 질문만
  prompt = f""" 너는 도로 관리 규정 안내 담당자다. 다음 질문에 두 문장 이내로 답하라.
                [질문] {question} """
  ans = call_llm([{"role": "user", "content": prompt}])
  return ans or "{LLM 응답 없음: 키를 확인하세요}"

def llm_rag(question): # RAG - 규정을 검색해 증거로 주입
  answer, evidence = raw_answer(question)
  return answer or "{LLM 응답 없음: 키를 확인하세요}", evidence

질문들 = [
    "밤에 신고하면 언제 처리 시작돼요?",
    "신고하면 보상금은 얼마나 받나요?",
]
for q in 질문들:
  print(f"질문: {q}\n")

  solo = _llm_solo(q)
  print(f"LLM 단독 (규정 없음): \n {solo}\n")

  rag, evidence = llm_rag(q)
  print(f"RAG (규정 주입):\n {rag}")
  print(f" [검색된 증거] {evidence[0][:45]}... \n" if evidence else "")

질문: 밤에 신고하면 언제 처리 시작돼요?

LLM 단독 (규정 없음): 
 야간에 접수된 신고는 다음 날 업무 시작 시간부터 순차적으로 처리가 시작됩니다. 긴급한 안전사고와 관련된 사항은 야간 당직반을 통해 신속히 조치하겠습니다.

RAG (규정 주입):
 [야간(22시~06시) 접수 신고는 익일 오전 담당자 배정 후 처리를 시작한다.] 밤에 접수된 신고는 익일 오전 담당자 배정 후 처리가 시작됩니다.
 [검색된 증거] ('야간(22시~06시) 접수 신고는 익일 오전 담당자 배정 후 처리를 시작한다.', 0.3061861991882324)... 

질문: 신고하면 보상금은 얼마나 받나요?

LLM 단독 (규정 없음): 
 도로 파손 등 안전신문고를 통한 신고에 대해선 현재 별도의 현금 보상 제도가 운영되지 않고 있습니다. 다만, 안전한 도로 환경 조성에 기여하는 공익적 가치가 큰 활동입니다.

RAG (규정 주입):
 규정에 없음.
 [검색된 증거] ('야간(22시~06시) 접수 신고는 익일 오전 담당자 배정 후 처리를 시작한다.', 0.3535533845424652)... 



In [6]:
print(chunks)

['제1조(긴급 보수) 심각도 4 이상의 포트홀은 접수 후 24시간 이내에 긴급 보수팀을 배정한다. 배정 결과는 신고자에게 통보한다.', '제2조(정기 처리) 심각도 3 이하의 도로 균열은 주간 정기 점검 일정에 포함하여 7일 이내 처리한다.', '제3조(맨홀) 맨홀 뚜껑 이상 신고는 안전 펜스를 우선 설치하고 상수도사업소에 이관한다. 이관 후에도 처리 현황을 추적한다.', '제4조(낙하물) 낙하물 신고는 2차 사고 예방을 위해 즉시 현장 출동을 원칙으로 한다.', '제5조(야간 접수) 야간(22시~06시) 접수 신고는 익일 오전 담당자 배정 후 처리를 시작한다. 단, 심각도 5는 야간에도 즉시 출동한다.', '제6조(완료 통보) 처리 완료된 신고는 사진과 함께 신고자에게 앱 알림으로 통보한다.']


In [7]:
print(chunk_vectors.shape)

torch.Size([6, 216])
